# Step 01: Establish Baseline Model
Loading green trip data, engineering features, and training a baseline regression model.

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
import warnings
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings('ignore')

In [ ]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)
    
    # Calculate duration in minutes
    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)
    
    # Filter duration to between 1 and 60 minutes
    df = df[(df.duration >= 1) & (df.duration <= 60)]
    
    # Create categorical PU_DO feature
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    
    return df

# Ensure you have downloaded the parquet files into a 'data' folder at the root of your project
data_path_train = '../data/green_tripdata_2023-01.parquet'
data_path_val = '../data/green_tripdata_2023-02.parquet'

try:
    df_train = read_dataframe(data_path_train)
    df_val = read_dataframe(data_path_val)
    print(f"Train records: {len(df_train)}, Validation records: {len(df_val)}")
except FileNotFoundError:
    print("Data not found. Please verify your relative paths to the D:\\ drive data folder.")

In [ ]:
categorical = ['PU_DO']
numerical = ['trip_distance']

dv = DictVectorizer()

# Fit DictVectorizer and transform training data
train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

# Transform validation data
val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

# Define target variable
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

# Train baseline Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)

# Predict and evaluate
y_pred = lr.predict(X_val)

mae = mean_absolute_error(y_val, y_pred)
rmse = mean_squared_error(y_val, y_pred, squared=False)

print(f"Validation MAE: {mae:.4f}")
print(f"Validation RMSE: {rmse:.4f}")

In [ ]:
os.makedirs('../models', exist_ok=True)

with open('../models/baseline.pkl', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)
    
print("Model successfully saved to ../models/baseline.pkl")